In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load all price chunks
files = sorted(Path('../data/raw/equities').glob('chunk_*.parquet'))
print(f"Found {len(files)} chunk files")

panel = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print(f"Total rows: {len(panel):,}")
print(f"Columns: {list(panel.columns)}")
print(f"\nFirst 5 rows:")
print(panel.head())

Found 33 chunk files
Total rows: 15,147,544
Columns: ['ticker', 'date', 'field', 'value']

First 5 rows:
               ticker        date      field   value
0  2261689D LN Equity  2016-03-04    PX_LAST  2000.0
1  2261689D LN Equity  2016-03-04    PX_OPEN  2000.0
2  2261689D LN Equity  2016-03-04    PX_HIGH  2000.0
3  2261689D LN Equity  2016-03-04     PX_LOW  2000.0
4  2261689D LN Equity  2016-03-04  PX_VOLUME     NaN


In [2]:
# Pivot from long to wide: one row per ticker-date, one column per field
panel_wide = panel.pivot_table(
    index=['ticker', 'date'],
    columns='field',
    values='value'
).reset_index()

# Flatten column names
panel_wide.columns.name = None

print(f"Wide format: {len(panel_wide):,} rows x {len(panel_wide.columns)} columns")
print(f"Columns: {list(panel_wide.columns)}")
print(f"Unique tickers: {panel_wide['ticker'].nunique()}")
print(f"Date range: {panel_wide['date'].min()} to {panel_wide['date'].max()}")
print(f"\nSample row:")
print(panel_wide.head(1).T)

Wide format: 1,893,443 rows x 10 columns
Columns: ['ticker', 'date', 'CUR_MKT_CAP', 'EQY_SH_OUT', 'PX_HIGH', 'PX_LAST', 'PX_LOW', 'PX_OPEN', 'PX_VOLUME', 'TOT_RETURN_INDEX_GROSS_DVDS']
Unique tickers: 639
Date range: 2009-07-01 to 2025-12-31

Sample row:
                                              0
ticker                       0966088D LN Equity
date                                 2014-03-28
CUR_MKT_CAP                                 NaN
EQY_SH_OUT                                  NaN
PX_HIGH                                     NaN
PX_LAST                                   180.0
PX_LOW                                      NaN
PX_OPEN                                     NaN
PX_VOLUME                                   NaN
TOT_RETURN_INDEX_GROSS_DVDS               180.0


In [4]:
# 3.1 Tickers per month
panel_wide['month'] = pd.to_datetime(panel_wide['date']).dt.to_period('M')
coverage = panel_wide.groupby('month')['ticker'].nunique()
print("=== TICKERS PER MONTH ===")
print(f"Min:    {coverage.min()}")
print(f"Max:    {coverage.max()}")
print(f"Median: {coverage.median()}")
print(f"Months below 200 tickers: {(coverage < 200).sum()}")

# 3.2 Field coverage — what % of rows have non-null values per field
print("\n=== FIELD COVERAGE (% non-null) ===")
for col in ['PX_LAST', 'PX_OPEN', 'PX_HIGH', 'PX_LOW', 'PX_VOLUME',
            'EQY_SH_OUT', 'CUR_MKT_CAP', 'TOT_RETURN_INDEX_GROSS_DVDS']:
    pct = panel_wide[col].notna().mean() * 100
    print(f"  {col:40s} {pct:6.1f}%")

# 3.3 Missing tickers — which 2 from master list have no data?
master = pd.read_parquet('../data/raw/master_tickers.parquet')
pulled = set(panel_wide['ticker'].unique())
missing = set(master['ticker']) - pulled
print(f"\n=== MISSING TICKERS ({len(missing)}) ===")
for t in sorted(missing):
    print(f"  {t}")

=== TICKERS PER MONTH ===
Min:    394
Max:    489
Median: 453.0
Months below 200 tickers: 0

=== FIELD COVERAGE (% non-null) ===
  PX_LAST                                    99.0%
  PX_OPEN                                    99.0%
  PX_HIGH                                    99.0%
  PX_LOW                                     99.0%
  PX_VOLUME                                  98.5%
  EQY_SH_OUT                                 80.2%
  CUR_MKT_CAP                                75.6%
  TOT_RETURN_INDEX_GROSS_DVDS                99.0%

=== MISSING TICKERS (2) ===
  1479704D LN Equity
  2067969D LN Equity


In [5]:
# 4.1 Daily returns from Total Return Index
panel_wide = panel_wide.sort_values(['ticker', 'date']).reset_index(drop=True)
panel_wide['ret'] = panel_wide.groupby('ticker')['TOT_RETURN_INDEX_GROSS_DVDS'].pct_change()

# Extreme returns (>50% in a single day)
extreme = panel_wide[panel_wide['ret'].abs() > 0.5].copy()
print(f"=== EXTREME DAILY RETURNS (|ret| > 50%) ===")
print(f"Count: {len(extreme)}")
if len(extreme) > 0:
    print(extreme[['date', 'ticker', 'PX_LAST', 'ret']].head(20).to_string())

# 4.2 Returns distribution
print(f"\n=== RETURN DISTRIBUTION ===")
print(panel_wide['ret'].describe())

# 4.3 Zero-volume days per ticker
zero_vol = (panel_wide.groupby('ticker')
            .apply(lambda d: (d['PX_VOLUME'] == 0).mean())
            .sort_values(ascending=False))
print(f"\n=== TICKERS WITH MOST ZERO-VOLUME DAYS ===")
print(f"Tickers with >20% zero-vol days: {(zero_vol > 0.2).sum()}")
print(zero_vol.head(10).to_string())

# 4.4 Sanity check: pick a known FTSE 250 stock
test_ticker = '3IN LN Equity'  # 3i Infrastructure — long-standing FTSE 250 member
test = panel_wide[panel_wide['ticker'] == test_ticker].sort_values('date')
print(f"\n=== SANITY CHECK: {test_ticker} ===")
print(f"Date range: {test.date.min()} to {test.date.max()}")
print(f"Trading days: {len(test)}")
print(f"Price range: {test.PX_LAST.min():.1f} to {test.PX_LAST.max():.1f}")
print(f"First price: {test.PX_LAST.iloc[0]:.1f}")
print(f"Last price: {test.PX_LAST.iloc[-1]:.1f}")

=== EXTREME DAILY RETURNS (|ret| > 50%) ===
Count: 147
             date              ticker  PX_LAST       ret
1605   2012-06-29  1245147D LN Equity  140.000  0.530055
13042  2014-04-14  1820495D LN Equity  530.000  0.549708
20069  2020-03-12  2448132D LN Equity    4.500 -0.799465
20070  2020-03-13  2448132D LN Equity   11.630  1.584444
58350  2015-01-27       AFR LN Equity    5.000 -0.717035
58354  2015-02-02       AFR LN Equity   10.000  0.886792
58451  2015-06-23       AFR LN Equity    2.390  0.593333
82708  2018-11-30       ALM LN Equity   64.500  0.563636
93194  2019-08-29      AMGO LN Equity   70.700 -0.516416
93407  2020-07-02      AMGO LN Equity    8.500  0.647328
93408  2020-07-03      AMGO LN Equity   13.720  0.614088
93633  2021-05-25      AMGO LN Equity    8.320 -0.553391
93832  2022-03-07      AMGO LN Equity    6.500  1.363127
94095  2023-03-23      AMGO LN Equity    0.250 -0.856100
94110  2023-04-17      AMGO LN Equity    0.275  0.569948
94111  2023-04-18      AMGO LN Eq

In [7]:
# Convert date column to proper datetime
panel_wide['date'] = pd.to_datetime(panel_wide['date'])

CUTOFF = pd.Timestamp('2024-01-01')

# Price panel
train = panel_wide[panel_wide['date'] < CUTOFF].copy()
holdout = panel_wide[panel_wide['date'] >= CUTOFF].copy()

# Benchmark
bench = pd.read_parquet('../data/raw/benchmark.parquet')
bench['date'] = pd.to_datetime(bench['date'])
bench_train = bench[bench['date'] < CUTOFF]
bench_holdout = bench[bench['date'] >= CUTOFF]

# SONIA
sonia = pd.read_parquet('../data/raw/sonia.parquet')
sonia['date'] = pd.to_datetime(sonia['date'])
sonia_train = sonia[sonia['date'] < CUTOFF]
sonia_holdout = sonia[sonia['date'] >= CUTOFF]

# Save training set
train.to_parquet('../data/working/equities_train.parquet')
bench_train.to_parquet('../data/working/benchmark_train.parquet')
sonia_train.to_parquet('../data/working/sonia_train.parquet')

# Save holdout
holdout.to_parquet('../data/holdout_locked/equities_holdout.parquet')
bench_holdout.to_parquet('../data/holdout_locked/benchmark_holdout.parquet')
sonia_holdout.to_parquet('../data/holdout_locked/sonia_holdout.parquet')

print("=== HOLDOUT SPLIT ===")
print(f"Cutoff: {CUTOFF.date()}")
print(f"\nTraining set:")
print(f"  Equities: {len(train):,} rows, {train['ticker'].nunique()} tickers")
print(f"  Date range: {train['date'].min().date()} to {train['date'].max().date()}")
print(f"  Benchmark: {len(bench_train):,} rows")
print(f"  SONIA: {len(sonia_train):,} rows")
print(f"\nHoldout set (DO NOT TOUCH UNTIL FINAL EVALUATION):")
print(f"  Equities: {len(holdout):,} rows, {holdout['ticker'].nunique()} tickers")
print(f"  Date range: {holdout['date'].min().date()} to {holdout['date'].max().date()}")
print(f"  Benchmark: {len(bench_holdout):,} rows")
print(f"  SONIA: {len(sonia_holdout):,} rows")

# Generate SHA-256 hashes for verification
import hashlib
print(f"\n=== HOLDOUT FILE HASHES (SHA-256) ===")
for fpath in sorted(Path('../data/holdout_locked').glob('*.parquet')):
    with open(fpath, 'rb') as f:
        h = hashlib.sha256(f.read()).hexdigest()
    print(f"  {fpath.name}: {h[:16]}...")

=== HOLDOUT SPLIT ===
Cutoff: 2024-01-01

Training set:
  Equities: 1,686,031 rows, 635 tickers
  Date range: 2009-07-01 to 2023-12-29
  Benchmark: 7,326 rows
  SONIA: 3,663 rows

Holdout set (DO NOT TOUCH UNTIL FINAL EVALUATION):
  Equities: 207,412 rows, 426 tickers
  Date range: 2024-01-01 to 2025-12-31
  Benchmark: 1,014 rows
  SONIA: 507 rows

=== HOLDOUT FILE HASHES (SHA-256) ===
  benchmark_holdout.parquet: 8208d8e90724f55d...
  equities_holdout.parquet: b4ab2336266d93a6...
  sonia_holdout.parquet: 207d27abd35c69d2...
